# Create round_info.csv and Dave config

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/prepare_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.dave import create_round_info, create_dave_config

In [7]:
SETTINGS_DIR  = SAMPLE_DIR / "settings"
METADATA_DIR  = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_NAME = SAMPLE_DIR.name
print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")

SAMPLE_DIR   : c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time
SAMPLE_NAME  : 251225_LT027_saving_time


In [ ]:
# ── Experiment parameters ──────────────────────────────────────────
MICROSCOPE           = "MF3"   # microscope identifier
N_HYBS               = 9       # number of hybridisation (bits) rounds
USE_ADAPTORS         = True    # True = adaptor-based fluidics; False = direct readouts
INCLUDE_FINAL_CLEAVE = False   # True = add a final cleave step after last imaging round
FIRST_HYB_NO_CLEAVE  = True    # True = first hyb (after the cells round) omits the cleave step

# Default round structure: imaging round 1 = cells only; rounds 2..N_HYBS+1 = bits #1..#N.
# The fluidics before the first bits round has no cleave; later fluidics include the cleave.

# HAL config filenames (from notebook 01 / SETTINGS_DIR)
# Adjust these to match the actual files created by notebook 01
bits_hal_configs  = sorted(SETTINGS_DIR.glob("hal-config-*bits*.xml"))
cells_hal_configs = sorted(SETTINGS_DIR.glob("hal-config-*cells*.xml"))

print("Available HAL configs in settings/:")
for p in sorted(SETTINGS_DIR.glob("hal-config-*.xml")):
    print(f"  {p.name}")

# Set these manually if auto-detection picks the wrong files
BITS_HAL_CONFIG  = bits_hal_configs[0].name  if bits_hal_configs  else "hal-config-mf3-bits.xml"
CELLS_HAL_CONFIG = cells_hal_configs[0].name if cells_hal_configs else "hal-config-mf3-cells.xml"

print(f"\nBits  HAL config : {BITS_HAL_CONFIG}")
print(f"Cells HAL config : {CELLS_HAL_CONFIG}")
print(f"\nUse adaptors        : {USE_ADAPTORS}")
print(f"Final cleave        : {INCLUDE_FINAL_CLEAVE}")
print(f"First hyb no cleave : {FIRST_HYB_NO_CLEAVE}")

In [9]:
round_info = create_round_info(
    microscope       = MICROSCOPE,
    n_bits           = N_HYBS,
    bits_hal_config  = BITS_HAL_CONFIG,
    cells_hal_config = CELLS_HAL_CONFIG,
    sample_dir       = SAMPLE_DIR,
)

print(round_info.to_string(index=False))

out_csv = METADATA_DIR / "round_info.csv"
round_info.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

 imaging_round                      series                                               hal_config                                                                                       data_dir
             1    hal-mf3-epi_01_{fov:03d} hal-config-mf3-bits-blkf4-488f2-560f25-650f25-750f25.xml       c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\data
             1 hal-mf3-epi-cells_{fov:03d}                    hal-config-mf3-cells-405f25-488f2.xml c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\data\cells
             2    hal-mf3-epi_02_{fov:03d} hal-config-mf3-bits-blkf4-488f2-560f25-650f25-750f25.xml       c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\data
             3    hal-mf3-epi_03_{fov:03d} hal-config-mf3-bits-blkf4-488f2-560f25-650f25-750f25.xml       c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\data
             4    hal-mf3

In [ ]:
positions_file = POSITIONS_DIR / f"positions_{SAMPLE_NAME}.txt"
if not positions_file.exists():
    raise FileNotFoundError(f"Positions file not found: {positions_file}")

dave_name   = f"dave-{MICROSCOPE.lower()}-{N_HYBS}hybs-{SAMPLE_NAME}.xml"
dave_output = SETTINGS_DIR / dave_name

create_dave_config(
    round_info           = round_info,
    positions_file       = positions_file,
    settings_dir         = SETTINGS_DIR,
    output_path          = dave_output,
    use_adaptors         = USE_ADAPTORS,
    include_final_cleave = INCLUDE_FINAL_CLEAVE,
    first_hyb_no_cleave  = FIRST_HYB_NO_CLEAVE,
)

print(f"Dave config saved: {dave_output}")

# Preview the generated file
with open(dave_output, encoding="ISO-8859-1") as fh:
    print(fh.read())

## Kilroy config

Looks for a Kilroy config for the current microscope in `MERci/data/configs/kilroy/`.
Files must contain the microscope name and end with a `YYMMDD` date stamp.
The newest matching file is copied to `SAMPLE_DIR/settings/`.

In [ ]:
import re, shutil

_kilroy_dir        = MERCI_DIR / "data" / "configs" / "kilroy"
_kilroy_candidates = [
    p for p in _kilroy_dir.glob("*.xml")
    if MICROSCOPE.lower() in p.name.lower()
]

if not _kilroy_candidates:
    print(f"No kilroy config found for microscope '{MICROSCOPE}' in {_kilroy_dir} — skipping.")
else:
    def _kilroy_date(p):
        m = re.search(r'(\d{6})\.xml$', p.name)
        return int(m.group(1)) if m else 0

    kilroy_src  = max(_kilroy_candidates, key=_kilroy_date)
    kilroy_dest = SETTINGS_DIR / kilroy_src.name
    shutil.copy2(str(kilroy_src), str(kilroy_dest))
    print(f"Kilroy config  : {kilroy_src.name}")
    print(f"Copied to      : {kilroy_dest}")